# Dashboard

The "Latest and Greatest" page of the old Power BI report, reworked in Altair from
`processed_data.processed_responses` (made by `processing/new_proccess_data.ipynb`). The subtitle also
shows when each `static_data` table was last fetched, from `static_data.fetched`.

The whole page is one Altair chart, so its filters run in the browser, also in the saved
`dashboard.html`:

- **All / Identified / Verified / Unique visitors** (top right): count every visit, visits with a student number,
  visits by students found in the student data, or each student once per bar.
- **Dates**: drag across the timeline; click the timeline outside the selection to remove it.
- **Semester, education, study start, gender, location, study line, AI use, department, course**: click a bar, shift-click to pick multiple.
  The other charts filter; the clicked one highlights. Click a picked bar again, or empty space in its chart, to
  remove it. Other study lines, every line past the top 20, is one bar split by education: hover an education to see
  its lines, click it to pick them all.
- **Filters** (top right) lists the filters in use: click one to switch it off or on, × removes it, Clear all
  removes them all.

Staff visits are left out: the dashboard covers students. Anonymous visits, from before the survey asked for a
student number, count only under All visitors, and the charts of student data (education, gender, study start, study
line, weekday by gender) leave them out.

In [ ]:
import json
import math
import os
from pathlib import Path
from string import Template

import altair as alt
import pandas as pd
import vl_convert
from pis_lib.connect import get_pg_engine

alt.data_transformers.disable_max_rows()  # the default 5000-row guard; every row is embedded in the page

In [ ]:
pg = get_pg_engine()

responses_df = pd.read_sql(
    'SELECT created_at, visitor_id, satisfaction, building_number, gender, "education_EN", "direction_DK",'
    " weekday, time_slot, semester, study_start, verified, ai, course, department"
    " FROM processed_data.processed_responses WHERE NOT staff",
    pg,
)
assert len(responses_df), "no student visits yet: nothing to draw"
fetched = pd.read_sql("SELECT table_name, fetched_at FROM static_data.fetched ORDER BY 1", pg)

## Prepare

- `week` and `month` are the start of each period, in Copenhagen time without a time zone, so every
  browser shows the same dates.
- Charts colored by education show the 8 largest groups, then `Other` (the rest) and `Unknown`, both gray, so no
  chart needs more than the 8 palette colors. Their legend, 4 to a line, lists the 5 largest by name, then the next
  3 by size, so the uncommon ones share the second line and `Other` and `Unknown` take the last.
- The student data comes with `Unknown` and `N/A` from `processing/new_proccess_data.ipynb`, and is NULL for
  anonymous visits, which have no student: the charts of student data leave them out.
- `verified` becomes 1 or 0, so charts can add it up.
- `course` drops the ` (Polytechnical Foundation)` ending most names have, so they fit their labels; the
  course codes keep names like `Mathematics 1a` apart.
- `CURRENT` is the semester of the latest visit, and `CURRENT_WEEK` the first week with a visit in it.
- `FETCHED` lists the day each `static_data` table was last fetched, in Copenhagen time.
- `AI_SINCE` and `COURSE_SINCE`, the first visits with an `ai` or a `course` value, are when each question
  was added. `course` keeps the 15 courses named most often, and is `No course` for visits since
  `COURSE_SINCE` that named none; other visits get no value.
- `department` is the department of the course named, without `Department of`: `01 Applied Mathematics and
  Computer Science`. `Unknown` when the course data has none, `No course` for visits since `COURSE_SINCE`
  that named none. Earlier visits get no value: the survey didn't ask yet.

In [ ]:
UNKNOWN = "Unknown"
NOT_APPLICABLE = "N/A"
NO_ANSWER = "No answer"
NO_COURSE = "No course"
OTHER = "Other"
OTHER_LINES = "Other study lines"
AI_ANSWERS = ["Yes", "No", NO_ANSWER]
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

responses_df["course"] = responses_df["course"].str.replace(r" \(Polytechnical foundation\)$", "", case=False, regex=True)
local_time = responses_df["created_at"].dt.tz_convert("Europe/Copenhagen").dt.tz_localize(None)
educations = responses_df.loc[~responses_df["education_EN"].isin([UNKNOWN, OTHER]), "education_EN"].value_counts().index
top_educations = [*sorted(educations[:5]), *educations[5:8]]
AI_SINCE = local_time[responses_df["ai"].notna()].min()
COURSE_SINCE = local_time[responses_df["course"].notna()].min()
top_courses = responses_df["course"].value_counts().index[:15]

df = (
    responses_df
    .assign(
        week=local_time.dt.to_period("W").dt.start_time,
        month=local_time.dt.to_period("M").dt.start_time,
        education_group=responses_df["education_EN"].where(
            responses_df["education_EN"].isin([*top_educations, UNKNOWN]) | responses_df["education_EN"].isna(), OTHER
        ),
        verified=responses_df["verified"].astype(int),
        course=responses_df["course"].where(responses_df["course"].isin(top_courses))
        .mask(responses_df["course"].isna() & (local_time >= COURSE_SINCE), NO_COURSE),
        department=responses_df["department"].str.replace(" Department of ", " ", regex=False)
        .fillna(UNKNOWN).where(responses_df["course"].notna(), NO_COURSE).where(local_time >= COURSE_SINCE),
    )
    .drop(columns="created_at")
)


def order(column, first=None):
    """The column's values in a fixed order (`first`, else sorted), Unknown and No course last."""
    present = set(df[column].dropna())
    last = [value for value in (UNKNOWN, NO_COURSE) if value in present]
    return [value for value in (sorted(present) if first is None else first) if value in present and value not in last] + last


SEMESTERS = order("semester", list(df.sort_values("month")["semester"].unique()))
EDUCATIONS = order("education_EN", list(df["education_EN"].value_counts().index))
EDUCATION_GROUPS = [*top_educations, *(group for group in (OTHER, UNKNOWN) if group in set(df["education_group"]))]
COURSES = order("course", list(top_courses))
DEPARTMENTS = order("department", list(df["department"].value_counts().index))
CURRENT = SEMESTERS[-1]
CURRENT_WEEK = df.loc[df["semester"] == CURRENT, "week"].min()
FETCHED = " · ".join(
    f"{row.table_name} {row.fetched_at.tz_convert('Europe/Copenhagen').strftime('%d %b %Y').lstrip('0')}" for row in fetched.itertuples()
) or "not recorded"

## Style

Colors are written in OKLCH: lightness from 0 to 1, chroma from 0 (gray) up, and hue in degrees. Equal steps
in it look equal, so a color is easy to make lighter or less saturated. `oklch()` turns each into the hex Vega draws:
vl-convert, which saves charts as PNG or SVG, draws `oklch(...)` colors black.

Each color has a light and a dark version, except blue, which keeps white text readable in both. `theme()` makes it
an expression on the `dark` param, which the page's script sets from the device's theme or the button, so the chart
changes theme in place and keeps its filters.

Colors follow the category, not its rank: each color scale lists every value, so a filter never
repaints the bars that remain. `Unknown`, `No answer` and `No course` are always light gray, the
education `Other` darker gray.

Dates, Semester and Cumulative show the current semester in blue; Dates and Semester show earlier ones in gray,
Cumulative in their season's color, orange for Fall and aqua for Spring. The bars you click to filter are gray too
until you pick one: then it turns blue, so blue means the current semester or a pick.

In [ ]:
def oklch(lightness, chroma, hue):
    """The sRGB hex of an OKLCH color, by Björn Ottosson's Oklab matrices; channels outside sRGB clip."""
    a, b = chroma * math.cos(math.radians(hue)), chroma * math.sin(math.radians(hue))
    l = (lightness + 0.3963377774 * a + 0.2158037573 * b) ** 3
    m = (lightness - 0.1055613458 * a - 0.0638541728 * b) ** 3
    s = (lightness - 0.0894841775 * a - 1.2914855480 * b) ** 3
    linear = (4.0767416621 * l - 3.3077115913 * m + 0.2309699292 * s,
              -1.2684380046 * l + 2.6097574011 * m - 0.3413193965 * s,
              -0.0041960863 * l - 0.7034186147 * m + 1.7076147010 * s)
    srgb = (12.92 * c if c <= 0.0031308 else 1.055 * c ** (1 / 2.4) - 0.055 for c in linear)
    return "#" + "".join(f"{round(min(max(c, 0), 1) * 255):02x}" for c in srgb)


assert (oklch(1, 0, 0), oklch(0, 0, 0), oklch(0.628, 0.2577, 29.23)) == ("#ffffff", "#000000", "#ff0000")

DARK = alt.param(name="dark", value=False)  # the page's script sets it before drawing, and when the theme changes


def theme(light, dark):
    """A color that follows the theme: a Vega expression on the `dark` param."""
    return alt.ExprRef(f"dark ? '{dark}' : '{light}'")


# Blue, orange, aqua, yellow, magenta, green, violet, red; the dark ones are a step brighter or deeper for a dark page.
# Blue is one for both: the white text of picked buttons and filters stays readable on it, 4.7:1 (WCAG AA asks 4.5).
BLUE = oklch(0.56, 0.1626, 255.5)  # the page's CSS uses it too
PALETTE = [
    BLUE,
    theme(oklch(0.671, 0.175, 40.6), oklch(0.622, 0.173, 40)),
    theme(oklch(0.669, 0.141, 162), oklch(0.621, 0.1283, 163.1)),
    theme(oklch(0.764, 0.1612, 75.1), oklch(0.67, 0.143, 73)),
    theme(oklch(0.716, 0.141, 357.4), oklch(0.622, 0.171, 1)),
    theme(oklch(0.529, 0.18, 142.5), oklch(0.621, 0.2073, 142.56)),
    theme(oklch(0.527, 0.197, 283.4), oklch(0.67, 0.145, 287)),
    theme(oklch(0.623, 0.191, 25), oklch(0.669, 0.159, 22)),
]
GRAY_HUE = 98  # a trace of warm yellow, in every gray
SURFACE_LIGHT, SURFACE_DARK = oklch(0.991, 0.001, GRAY_HUE), oklch(0.217, 0.002, GRAY_HUE)  # the page's CSS uses these
INK_LIGHT, INK_DARK = oklch(0.15, 0, GRAY_HUE), oklch(0.952, 0.004, GRAY_HUE)  # and these
SURFACE = theme(SURFACE_LIGHT, SURFACE_DARK)
INK = theme(INK_LIGHT, INK_DARK)
INK_2 = theme(oklch(0.435, 0.005, GRAY_HUE), oklch(0.812, 0.015, GRAY_HUE))
MUTED = oklch(0.623, 0.009, GRAY_HUE)  # the same in both: it reads on either surface
GRID = theme(oklch(0.906, 0.009, GRAY_HUE), oklch(0.292, 0.004, GRAY_HUE))
AXIS = theme(oklch(0.812, 0.015, GRAY_HUE), oklch(0.34, 0.005, GRAY_HUE))
UNKNOWN_FILL = theme(oklch(0.812, 0.015, GRAY_HUE), oklch(0.41, 0.01, GRAY_HUE))
EARLIER = MUTED


def color(column, values=None, gray=()):
    """Palette colors in `values`' order (default: `order(column)`); Unknown and No answer light gray, `gray` darker."""
    domain = values or order(column)
    known = [value for value in domain if value not in (UNKNOWN, NO_ANSWER, *gray)]
    assert len(known) <= len(PALETTE), f"{column} has more values than colors; fold the rest into Other"
    palette = iter(PALETTE)
    return alt.Color(
        f"{column}:N", title=None, sort=domain,
        scale=alt.Scale(domain=domain, range=[
            UNKNOWN_FILL if value in (UNKNOWN, NO_ANSWER) else MUTED if value in gray else next(palette) for value in domain
        ]),
    )

## Filters

Every chart filters by all the selections except its own; its own one highlights instead, as in Power BI.
The page lists the filters in use under Filters, top right, to switch off and on, remove, or clear all at once
(`dashboard.js`, through the selections' stores). The toggle is a selection too, but no filter: `MODE` is
the picked button,
"All visitors" if none is.

In [ ]:
def pick(name, *fields, tested="datum"):
    """A click filter: clicking a picked value again removes it (toggles), shift-click adds or removes one.

    `tested` is what is checked for being picked already: the clicked datum, unless given.
    """
    return alt.selection_point(
        fields=list(fields), clear=False, name=name,  # the page's list of filters clears them
        toggle=f"event.shiftKey || (datum && vlSelectionTest('{name}_store', {tested}))",
    )


dates = alt.selection_interval(encodings=["x"], clear=False, name="dates")
semester_pick = pick("semester_pick", "semester")
education_pick = pick("education_pick", "education_EN")
start_pick = pick("start_pick", "study_start")
gender_pick = pick("gender_pick", "gender")
location_pick = pick("location_pick", "building_number")
# One bar segment, or a part of Other study lines: its education_EN and direction_DK are lists, and the pick matches
# rows with any of them. A list can't be checked for being picked, so the part is checked by its first education and
# line.
LINE_TESTED = "(isArray(datum.direction_DK) ? {education_EN: datum.education_EN[0], direction_DK: datum.direction_DK[0]} : datum)"
line_pick = pick("line_pick", "education_EN", "direction_DK", tested=LINE_TESTED)
ai_pick = pick("ai_pick", "ai")
department_pick = pick("department_pick", "department")
course_pick = pick("course_pick", "course")
FILTERS = [dates, semester_pick, education_pick, start_pick, gender_pick, location_pick, line_pick, ai_pick,
           department_pick, course_pick]
# What the page's list of filters calls each.
FILTER_LABELS = {
    "dates": "Dates", "semester_pick": "Semester", "education_pick": "Education", "start_pick": "Study start",
    "gender_pick": "Gender", "location_pick": "Location", "line_pick": "Study line", "ai_pick": "Tried AI",
    "department_pick": "Department", "course_pick": "Course",
}

visitor_mode = alt.selection_point(
    fields=["mode"], value=[{"mode": "All visitors"}], toggle=False, clear=False, name="visitor_mode"
)
MODES = {  # button: what it counts, shown on hover
    "All visitors": "Every visit, including anonymous ones. A student who comes back counts each time.",
    "Identified visitors": "Visits with a student number. A student who comes back counts each time.",
    "Verified visitors": "Visits by students found in the student data. A student who comes back counts each time.",
    "Unique visitors": "Each student once per bar or slice. Anonymous visits are left out.",
}
MODE = "((visitor_mode.mode && visitor_mode.mode[0]) || 'All visitors')"

base = alt.Chart(df)
ai_base = base.transform_filter("isValid(datum.ai)")
CLICK = "Click a bar to filter the other charts, again to remove it, shift-click for multiple"


def visitors(chart, by, own=None):
    """Filter by every selection except `own`, then count per `by`.

    `visitors` is what the toggle picks: all `visits`, `identified` visits (with a student number), `verified`
    visits (student found in the student data), or `unique` students.
    """
    for selection in FILTERS:
        if selection is not own:
            chart = chart.transform_filter(selection)
    return chart.transform_aggregate(
        visits="count()", students="distinct(visitor_id)", anonymous="missing(visitor_id)", verified="sum(verified)",
        rating="mean(satisfaction)", groupby=by,
    ).transform_calculate(
        identified="datum.visits - datum.anonymous",
        unique="datum.students - (datum.anonymous > 0 ? 1 : 0)",  # distinct() counts null as one more value
        visitors=f"{MODE} == 'Unique visitors' ? datum.unique : {MODE} == 'Verified visitors' ? datum.verified"
                 f" : {MODE} == 'Identified visitors' ? datum.identified : datum.visits",
    )


def tooltip(*fields):
    return [*fields, alt.Tooltip("visits:Q", title="All visitors"), alt.Tooltip("identified:Q", title="Identified visitors"),
            alt.Tooltip("verified:Q", title="Verified visitors"), alt.Tooltip("unique:Q", title="Unique visitors"),
            alt.Tooltip("rating:Q", title="Average rating", format=".2f")]


def highlight(selection):
    return alt.condition(selection, alt.value(1), alt.value(0.3))

## Charts

In [ ]:
# shortcut: the page is built PAGE_WIDTH wide, a little past the slanted labels of AI use by course, so the rows this
# wide are the widest on the page and dashboard.js's extra width ends at the window's edge. Change it, and timeline to
# match, if a section gets wider.
PAGE_WIDTH = 1420


def kpi(title, field, number_format):
    return visitors(base, []).mark_text(fontSize=30, align="left", color=INK).encode(
        x=alt.value(0), text=alt.Text(f"{field}:Q", format=number_format),
    ).properties(title=alt.Title(title, fontSize=12, color=INK_2, fontWeight="normal"), width=110, height=36)


kpis = alt.hconcat(kpi("Average rating", "rating", ".2f"), kpi("All visitors", "visits", ","),
                   kpi("Identified visitors", "identified", ","), kpi("Verified visitors", "verified", ","),
                   kpi("Unique visitors", "unique", ","))

toggle_base = alt.Chart(alt.InlineData(values=[{"mode": m, "description": d} for m, d in MODES.items()])).encode(
    x=alt.X("mode:N", axis=None, sort=list(MODES)),
)
picked = f"{MODE} == datum.mode"  # this button's mode is the one in use
toggle = alt.layer(
    toggle_base.mark_rect(cornerRadius=6, stroke=SURFACE, strokeWidth=6, cursor="pointer").encode(
        color=alt.condition(picked, alt.value(PALETTE[0]), alt.value(GRID)),
        tooltip="description:N",
    ).add_params(visitor_mode),
    toggle_base.mark_text(fontSize=13, cursor="pointer").encode(
        text="mode:N", color=alt.condition(picked, alt.value("white"), alt.value(INK_2)),
    ),
).properties(width=125 * len(MODES), height=40, title=alt.Title("Count", fontSize=12, color=INK_2, fontWeight="normal"))

# The page lists the filters in use here (dashboard.js). Vega-Lite only draws the title and leaves the room,
# about the rest of the row after the KPIs (5 of 110, 20 apart) and the Count buttons, 30 apart: the page makes it end
# where the dividers do, as the KPIs' text pushes it a little right, and as tall as the list.
filters_slot = alt.Chart(alt.InlineData(values=[{}])).mark_point(opacity=0).properties(
    name="filters", width=PAGE_WIDTH - (5 * 110 + 4 * 20) - 125 * len(MODES) - 2 * 30, height=40,
    title=alt.Title("Filters", fontSize=12, color=INK_2, fontWeight="normal"),
)

# The charts that widen with the window, by name, and their share of the width beyond what the page was built
# for; dashboard.js does the widening. Each section's divider adds itself.
GROW = {"timeline": 1, "cumulative": 1, "returning": 1}

# Each chart with a y axis keeps it inside, in its first pixels, so its title starts above the axis labels instead of
# right of them; Vega-Lite lines up the plots, not the axes. The page's script measures each axis, in the viewer's
# font, and sets its width; the guesses given here hold until then, and in this notebook.
AXES = {}  # chart name: the param with its y axis's width
AXIS_PARAMS = {}  # by name: the Dashboard cell calls bars() again on each run


def axis_room(name, guess, charts=None):
    """A param `<name>_axis` for the width of the y axis of chart `name`, or of `charts`, which share it.

    Returns the y axis's offset and the start of the x range, both following it.
    """
    room = alt.param(name=f"{name}_axis", value=guess)
    AXIS_PARAMS[room.name] = room
    AXES.update(dict.fromkeys(charts or [name], room.name))
    return alt.ExprRef(f"-{room.name}"), alt.ExprRef(room.name)

# A fixed date range: Vega-Lite's brush breaks ("dates_week is undefined") when its axis rescales with no brush drawn.
# With x2, Vega-Lite then logs 'Conflicting scale property "domain"'; harmless, both domains are this one.
first_week, last_week = df["week"].min(), df["week"].max() + pd.Timedelta(weeks=1)
y_offset, x_start = axis_room("timeline", 28)
timeline = visitors(base, ["week"], own=dates).transform_calculate(
    week_end="timeOffset('week', datum.week, 1)",
).mark_rect(stroke=SURFACE, strokeWidth=0.5).encode(
    color=alt.condition(alt.FieldGTEPredicate(field="week", gte=alt.DateTime(
        year=CURRENT_WEEK.year, month=CURRENT_WEEK.month, date=CURRENT_WEEK.day,
    )), alt.value(PALETTE[0]), alt.value(EARLIER)),
    # Labels within 40px of an end, over half the widest, end at their tick: a last label sticking out past the plot,
    # by more or less as the chart widens, would leave a gap at the window's edge or overflow it.
    x=alt.X("week:T", title=None, axis=alt.Axis(labelFlush=40), scale=alt.Scale(domain=[
        alt.DateTime(year=d.year, month=d.month, date=d.day) for d in (first_week, last_week)
    ], rangeMin=x_start)),
    x2="week_end:T",
    y=alt.Y("visitors:Q", title=None, axis=alt.Axis(offset=y_offset)), y2=alt.datum(0),
    tooltip=tooltip(alt.Tooltip("week:T", title="Week of", format="%-d %b %Y")),
).add_params(dates).properties(
    name="timeline", width=1160, height=180, title="Dates",
)


def bars(column, pick, title, values, inner_labels, width=200, subtitle=alt.Undefined, current=None):
    """Horizontal bars per category that double as a slicer: click one to filter the other charts.

    The labels sit inside the chart (see axis_room), cut at `inner_labels` - 10 pixels; the chart is `inner_labels` +
    `width` wide, and the bars take what the labels leave.
    Bars are gray; picked ones blue, and `current` too. Visits without a value are left out.
    """
    name = f"{column}_bars"
    y_offset, x_start = axis_room(name, inner_labels)
    fill = alt.when(pick, empty=False).then(alt.value(PALETTE[0])).when(
        f"indexof({[UNKNOWN, NO_ANSWER, NO_COURSE]}, datum.{column}) >= 0",
    ).then(alt.value(UNKNOWN_FILL))
    if current:
        fill = fill.when(f"datum.{column} == '{current}'").then(alt.value(PALETTE[0]))
    return visitors(base.transform_filter(f"isValid(datum.{column})"), [column], own=pick).mark_bar(cornerRadiusEnd=4, cursor="pointer").encode(
        color=fill.otherwise(alt.value(MUTED)),
        y=alt.Y(f"{column}:N", sort=values, title=None, axis=alt.Axis(offset=y_offset, labelLimit=inner_labels - 10)),
        x=alt.X("visitors:Q", title=None, scale=alt.Scale(rangeMin=x_start),
                axis=alt.Axis(tickCount=width // 40)),  # Vega-Lite would count the label space too
        opacity=highlight(pick),
        tooltip=tooltip(alt.Tooltip(f"{column}:N", title=title)),
    ).add_params(pick).properties(
        name=name, width=inner_labels + width, height=alt.Step(18), title=alt.Title(title, subtitle=subtitle),
    )


TIME_SLOTS = ["Morning", "Afternoon"]


def by_gender(column, column_title, values, title, y_axis, slots=False):
    """Bars per `column` value, stacked by gender; with `slots`, a panel per time slot, named by its y axis's title.

    `y_axis` is a guess at the y axis's width, as in axis_room.
    """
    names = [f"{column}_{slot.lower()}" for slot in TIME_SLOTS] if slots else [column]
    y_offset, x_start = axis_room(column, y_axis, names)

    def panel(name, slot=None, x_labels=True):
        students = base.transform_filter("isValid(datum.gender)")  # not anonymous visits, which have no student
        chart = students.transform_filter(f"datum.time_slot == '{slot}'") if slot else students
        return visitors(chart, [column, "gender"] + (["time_slot"] if slot else [])).transform_calculate(
            gender_order=f"indexof({order('gender')}, datum.gender)",
        ).mark_bar(stroke=SURFACE, strokeWidth=1).encode(
            # A slot per value, also those a filter empties, so the bars stay 60px apart, as alt.Step(60) would keep them.
            x=alt.X(f"{column}:N", title=None, axis=alt.Axis(labelAngle=0) if x_labels else None,
                    scale=alt.Scale(domain=values, rangeMin=x_start)),
            # A label on the plot's top edge hangs below its gridline, as in returning_chart, not above, lifting the title.
            y=alt.Y("visitors:Q", title=slot, axis=alt.Axis(offset=y_offset, labelFlush=True)),
            color=color("gender"),
            order=alt.Order("gender_order:Q"),  # Unknown on top
            tooltip=tooltip(alt.Tooltip(f"{column}:N", title=column_title), alt.Tooltip("gender:N", title="Gender"),
                            *([alt.Tooltip("time_slot:N", title="Time slot")] if slot else [])),
        ).properties(name=name, width=y_axis + 60 * len(values), height=100 if slot else 220)

    if not slots:
        return panel(column).properties(title=title)
    # Panels, not a facet: a facet's row labels sit outside the chart, where the page can't fit them to the axis.
    return alt.vconcat(
        panel(names[0], TIME_SLOTS[0], x_labels=False), panel(names[1], TIME_SLOTS[1]), spacing=12, title=title,
    ).resolve_scale(y="shared")


# A blank subtitle, as Location has: it lines their titles up with that of Tried fixing with AI before, beside them,
# whose subtitle lifts it. Their section title sits as far above all three.
weekday_chart = by_gender("weekday", "Weekday", order("weekday", WEEKDAYS),
                          alt.Title("Visitors by weekday and gender", subtitle=" "), y_axis=52, slots=True)
# Two blank subtitle lines, as Visitors by study line and education beside it has two: their titles line up.
start_chart = by_gender("study_start", "Study start", order("study_start"),
                        alt.Title("Visitors by study start and gender", subtitle=[" ", " "]), y_axis=36)

kinds = ["Unique visitors", "Repeat visits", "Anonymous visits"]
y_offset, x_start = axis_room("returning", 32)
returning_chart = visitors(base, ["month"]).transform_calculate(
    **{"Unique visitors": "datum.unique", "Repeat visits": "datum.visits - datum.unique - datum.anonymous",
       "Anonymous visits": "datum.anonymous"},
).transform_fold(kinds, as_=["kind", "count"]).mark_bar(stroke=SURFACE, strokeWidth=1).encode(
    x=alt.X("yearmonth(month):T", title=None, axis=alt.Axis(labelFlush=40),  # as in timeline
            scale=alt.Scale(rangeMin=x_start)),
    # 100% hangs below its gridline, inside the plot: above it, it would lift the title out of line with Cumulative's.
    y=alt.Y("count:Q", stack="normalize", title=None, axis=alt.Axis(format="%", labelFlush=True, offset=y_offset)),
    color=alt.Color("kind:N", title=None, sort=kinds, scale=alt.Scale(domain=kinds, range=[PALETTE[0], PALETTE[1], MUTED])),
    order=alt.Order("kind_order:Q"),
    tooltip=[alt.Tooltip("yearmonth(month):T", title="Month"), alt.Tooltip("kind:N", title=" "), alt.Tooltip("count:Q", title="Visits")],
).transform_calculate(kind_order=f"indexof({kinds}, datum.kind)").properties(
    name="returning", width=PAGE_WIDTH, height=200,
    title=alt.Title("Unique, repeat and anonymous visits by month", subtitle="Repeat: a student's 2nd or later visit that month"),
)

EARLIER_FALL = "Earlier Fall semesters"
EARLIER_SPRING = "Earlier Spring semesters"
y_offset, x_start = axis_room("cumulative", 44)
# As in Power BI: a running sum of each month's count, so a student who comes in two months counts twice.
# Months in calendar order: Spring semesters (January to June) on the left, Fall ones (September to December) on the
# right, so the semesters of a season line up month by month.
cumulative_base = visitors(base, ["semester", "month"]).transform_window(
    so_far="sum(visitors)", groupby=["semester"], sort=[alt.SortField("month")],
).encode(
    x=alt.X("month(month):O", title=None, axis=alt.Axis(labelAngle=0),
            scale=alt.Scale(rangeMin=x_start)),
    y=alt.Y("so_far:Q", title="Visitors so far", axis=alt.Axis(offset=y_offset)),
)
cumulative = alt.layer(
    cumulative_base.transform_calculate(
        period=f"datum.semester == '{CURRENT}' ? '{CURRENT}' "
               f": indexof(datum.semester, 'Fall') == 0 ? '{EARLIER_FALL}' : '{EARLIER_SPRING}'",
    ).mark_line(strokeWidth=2, point=alt.OverlayMarkDef(size=64, filled=True)).encode(
        # The current semester blue; each season its own color, whichever season is current.
        color=alt.Color("period:N", title=None, scale=alt.Scale(
            domain=[CURRENT, EARLIER_FALL, EARLIER_SPRING], range=[PALETTE[0], PALETTE[1], PALETTE[2]],
        )),
        detail="semester:N",
        tooltip=[alt.Tooltip("semester:N", title="Semester"), alt.Tooltip("yearmonth(month):T", title="Month"),
                 alt.Tooltip("visitors:Q", title="This month"), alt.Tooltip("so_far:Q", title="Semester so far")],
    ),
    cumulative_base.transform_window(  # each line's name, after its last month
        newest="rank()", groupby=["semester"], sort=[alt.SortField("month", order="descending")],
    ).transform_filter("datum.newest == 1").mark_text(align="left", dx=10, fontSize=11).encode(
        text="semester:N", color=alt.condition(f"datum.semester == '{CURRENT}'", alt.value(INK), alt.value(MUTED)),
    ),
).properties(name="cumulative", width=PAGE_WIDTH, height=220, title=alt.Title(
    "Cumulative visitors per semester",
    subtitle="Adds up each month's count; Spring semesters on the left, Fall on the right, to compare like with like",
))

# Study line names cut at LINE_LABELS - 10 pixels, as in bars().
TOP_LINES, LINE_LABELS, LINE_WIDTH = 20, 180, 440
LINE_STEP = 16  # each study line's row
y_offset, x_start = axis_room("study_lines", LINE_LABELS)
student_lines = base.transform_filter("isValid(datum.direction_DK)")
OTHER_EDUCATIONS = sorted(df.loc[df["education_group"] == OTHER, "education_EN"].unique())
line_y = alt.Y("line:N", sort=alt.EncodingSortField(field="line_rank", op="min"), title=None,
               axis=alt.Axis(offset=y_offset, labelLimit=LINE_LABELS - 10))
line_x = alt.X("visitors:Q", title=None, scale=alt.Scale(rangeMin=x_start), axis=alt.Axis(tickCount=LINE_WIDTH // 40))
# Both layers rank the lines by visitors, then name, so each line is in exactly one of them. N/A and Unknown rank
# after them all, so they take no place in the top and get the last rows, in LAST_LINES' order, under Other study
# lines.
LAST_LINES = [NOT_APPLICABLE, UNKNOWN]
top_lines = visitors(student_lines, ["education_group", "education_EN", "direction_DK"], own=line_pick).transform_joinaggregate(
    line_visitors="sum(visitors)", groupby=["direction_DK"],
).transform_calculate(
    last_line=f"indexof({LAST_LINES}, datum.direction_DK) + 1",  # 0 for the other lines
).transform_window(  # one rank per study line, as each line has a segment per education
    line_rank="dense_rank()",
    sort=[alt.SortField("last_line"), alt.SortField("line_visitors", order="descending"), alt.SortField("direction_DK")],
).transform_filter(f"datum.line_rank <= {TOP_LINES} || datum.last_line").transform_calculate(
    line="datum.direction_DK", line_rank=f"datum.last_line ? {TOP_LINES + 1} + datum.last_line : datum.line_rank",
).mark_bar(stroke=SURFACE, strokeWidth=1, cursor="pointer").encode(
    y=line_y, x=line_x,
    color=color("education_group", EDUCATION_GROUPS, gray=[OTHER]),
    opacity=highlight(line_pick),
    tooltip=tooltip(alt.Tooltip("education_EN:N", title="Education"), alt.Tooltip("direction_DK:N", title="Study line")),
).add_params(line_pick)
# One bar, a part per education, which picks the part's lines (see line_pick). Each part's tooltip lists them:
# dashboard.js draws `lines`, {rows: [[line, visitors], ...]}, as a row per line. Vega-Lite would join a
# bare array into one text.
other_lines = visitors(
    student_lines.transform_filter(f"indexof({LAST_LINES}, datum.direction_DK) < 0"), ["education_group", "direction_DK"], own=line_pick,
).transform_joinaggregate(
    line_visitors="sum(visitors)", groupby=["direction_DK"],
).transform_window(
    line_rank="dense_rank()", sort=[alt.SortField("line_visitors", order="descending"), alt.SortField("direction_DK")],
).transform_filter(f"datum.line_rank > {TOP_LINES}").transform_calculate(
    entry="[datum.direction_DK, datum.visitors]", rating_sum="datum.rating * datum.visits",
).transform_aggregate(
    visits="sum(visits)", identified="sum(identified)", verified="sum(verified)", unique="sum(unique)",
    visitors="sum(visitors)", rating_sum="sum(rating_sum)", entries="values(entry)", groupby=["education_group"],
).transform_calculate(
    line=f"'{OTHER_LINES}'", line_rank=f"{TOP_LINES + 1}",
    rating="datum.rating_sum / datum.visits",  # every visit has a rating
    lines="{rows: pluck(datum.entries, 'entry')}",
    education_EN=f"datum.education_group == '{OTHER}' ? {json.dumps(OTHER_EDUCATIONS, ensure_ascii=False)}"
                 " : [datum.education_group]",
    direction_DK="sort(pluck(datum.entries, 'direction_DK'))",  # as the list of filters shows them
).mark_bar(stroke=SURFACE, strokeWidth=1, cursor="pointer").encode(
    y=line_y, x=line_x,
    color=color("education_group", EDUCATION_GROUPS, gray=[OTHER]),
    opacity=alt.condition(f"!length(data('line_pick_store')) || vlSelectionTest('line_pick_store', {LINE_TESTED})",
                          alt.value(1), alt.value(0.3)),
    tooltip=tooltip(alt.Tooltip("education_group:N", title="Education"), alt.Tooltip("line:N", title="Study line"))
    + [alt.Tooltip("lines:N", title="Study lines")],
)
education_chart = alt.layer(top_lines, other_lines).properties(
    name="study_lines", width=LINE_LABELS + LINE_WIDTH, height=alt.Step(LINE_STEP),
    title=alt.Title("Visitors by study line and education", subtitle=[
        "Visits with a student number; Unknown: not in the student data, or no study line there",
        f"The {TOP_LINES} study lines with the most visitors; the rest are {OTHER_LINES}",
    ]),
)

AI_TITLE = "Tried fixing with AI before"
ai_since = f"Since the question was added, {AI_SINCE.day} {AI_SINCE:%b %Y}" if pd.notna(AI_SINCE) else "No answers yet"
# AI use by course: AI_AXIS pixels for its y axis (a guess, see axis_room), its bars the next AI_WIDTH. The AI bars
# above are as wide in all, so both end alike.
AI_AXIS, AI_WIDTH = 45, max(28 * len(COURSES), 180)
ai_chart = bars("ai", ai_pick, AI_TITLE, AI_ANSWERS, width=AI_AXIS + AI_WIDTH - 80, subtitle=ai_since,
                inner_labels=80)
course_since = f"since the question was added, {COURSE_SINCE.day} {COURSE_SINCE:%b %Y}" if pd.notna(COURSE_SINCE) else "no answers yet"
course_bars = bars("course", course_pick, "Course", COURSES, inner_labels=180,
                   subtitle=[f"The 15 courses named most often, {course_since}", "No course: visits that named none"])
department_bars = bars("department", department_pick, "Department", DEPARTMENTS, width=160, inner_labels=190,
                       subtitle=[f"Of the course named, {course_since}", "No course: visits that named none"])

# Only highlights the course picked in course_bars, the one course slicer.
y_offset, x_start = axis_room("ai_course", AI_AXIS)
ai_course_chart = visitors(ai_base.transform_filter("isValid(datum.course)"), ["course", "ai"], own=course_pick).transform_joinaggregate(
    course_visitors="sum(visitors)", groupby=["course"],
).transform_calculate(
    share="datum.visitors / datum.course_visitors", ai_order=f"indexof({AI_ANSWERS}, datum.ai)",
).mark_bar(stroke=SURFACE, strokeWidth=2).encode(
    # Labels slant right: names hanging left of the plot would push every other row right.
    # A fixed domain keeps a slot per course, so the width stays AI_WIDTH under every filter.
    x=alt.X("course:N", scale=alt.Scale(domain=COURSES, rangeMin=x_start), title=None,
            axis=alt.Axis(labelAngle=45, labelLimit=180)),
    y=alt.Y("visitors:Q", title="Visitors", axis=alt.Axis(offset=y_offset)),
    color=color("ai", AI_ANSWERS),
    order=alt.Order("ai_order:Q"),
    opacity=highlight(course_pick),
    tooltip=tooltip(alt.Tooltip("course:N", title="Course"), alt.Tooltip("ai:N", title=AI_TITLE),
                    alt.Tooltip("share:Q", title="Share of the course", format=".0%")),
).properties(
    name="ai_course", width=AI_AXIS + AI_WIDTH, height=220,
    title=alt.Title("AI use by course", subtitle=["The 15 courses named most often", ai_since]),
)

## Dashboard

Each chart keeps its own colors and legend (`independent`); by default Altair merges them into one. Legends
sit under their charts, untitled: the chart title or the labels say what they show. How to filter is said once, under the page title.

The charts are grouped in sections: Period, Demographics (who visits), Visit (where, when, for which course,
with or without AI) and Trends (visits over the months and semesters).

In [ ]:
def row(*charts):
    return alt.hconcat(*charts, spacing=30).resolve_scale(color="independent")


def column(*charts):
    return alt.vconcat(*charts, spacing=30).resolve_scale(color="independent")


def section(title, *rows):
    """A hairline across the page, then the titled charts."""
    name = f"{title.lower()}_divider"
    GROW[name] = 1
    divider = alt.Chart(alt.InlineData(values=[{}])).mark_rule(color=AXIS).encode(
        x=alt.value(0), x2=alt.value("width"), y=alt.value(0),
    ).properties(name=name, width=PAGE_WIDTH, height=1)
    charts = column(*rows).properties(title=alt.Title(title, fontSize=20, offset=24))
    return alt.vconcat(divider, charts, spacing=16).resolve_scale(color="independent")


built = pd.Timestamp.now(tz="Europe/Copenhagen")
dashboard = alt.vconcat(
    row(kpis, toggle, filters_slot),
    section("Period", row(timeline, bars("semester", semester_pick, "Semester", SEMESTERS, width=160, inner_labels=70, current=CURRENT))),
    section(
        "Demographics",
        row(bars("education_EN", education_pick, "Education", EDUCATIONS, inner_labels=120),
            bars("gender", gender_pick, "Gender", order("gender"), width=160, inner_labels=60),
            bars("study_start", start_pick, "Study start", order("study_start"), inner_labels=85)),
        # As tall as the study lines beside it: TOP_LINES, Other study lines, N/A and Unknown. dashboard.js keeps it so
        # as a filter changes how many lines there are.
        row(education_chart, start_chart.properties(height=(TOP_LINES + 3) * LINE_STEP)),
    ),
    section(
        "Visit",
        row(column(row(weekday_chart, bars("building_number", location_pick, "Location", order("building_number"), width=160,
                                           subtitle=" ", inner_labels=55)),
                   row(department_bars, course_bars)),
            column(ai_chart, ai_course_chart)),
    ),
    section("Trends", cumulative, returning_chart),
    spacing=72,
).add_params(DARK, *AXIS_PARAMS.values()).resolve_scale(color="independent").properties(
    padding=64,  # the page's margin, all around; dashboard.js lines its buttons up with it
    title=alt.Title("Python Installation Support statistics", fontSize=24, subtitle=[
        f"{len(df):,} visits, {df['month'].min():%b %Y} to {df['month'].max():%b %Y} · "
        # Old during opening hours (hourly runs) means the pipeline is failing.
        f"built {built.day} {built:%b %Y %H:%M}",
        f"Data fetched: {FETCHED}",
        CLICK,
        "Drag across Dates to filter by week, click outside the selection to remove it",
    ]),
).configure(
    background=SURFACE, font="system-ui, -apple-system, 'Segoe UI', sans-serif",
).configure_view(stroke=None).configure_title(
    # Titles start where the plots do, not at axis labels hanging left of them, so they line up down the page.
    anchor="start", frame="group", color=INK, subtitleColor=MUTED, fontSize=14, fontWeight=500,
).configure_axis(
    domainColor=AXIS, tickColor=AXIS, gridColor=GRID, labelColor=MUTED, titleColor=INK_2, titleFontWeight="normal",
    labelSeparation=8,  # on time and number axes, every other label goes when two would touch, as wide ones do
).configure_legend(orient="bottom", columns=4, labelColor=INK_2)

dashboard

Save the page: `dashboard_template.html`, filled in with its style (`dashboard.css`) and script (`dashboard.js`), the
chart, a copy of Vega (the page loads nothing from a CDN), `GROW` and `AXES`. The script fits the y axis of each chart in `AXES` to its labels, then widens the charts in `GROW` while the window is wider than the page was built: each
takes its share of the extra width, through its width signal (`timeline_width`, found by the chart's name), until the
dividers end as far from the window's right edge as they start from its left.

The page embeds every row of `df` (no student numbers or names), so share it only with people who
may see the survey data, and don't commit it.

`DASHBOARD_HTML` sets where it goes instead. On the server, the pipeline saves it outside the checkout, and the
admin shows it at `/admin/dashboard/`.

In [ ]:
class ScriptSafeJSON(json.JSONEncoder):
    """JSON that can't close the page's <script> tag: every < becomes \\u003c, which reads back the same."""

    def encode(self, o):
        return super().encode(o).replace("<", "\\u003c")


def width_signals(marks):
    """Each named chart's width signal, from its group mark. Vega-Lite merges the signals of charts as wide as each
    other in one column, into one named after the column: Trends' divider and both its charts share one."""
    found = {}
    for mark in marks:
        width = mark.get("encode", {}).get("update", {}).get("width", {})
        if mark.get("name", "").endswith("_group") and "signal" in width:
            found[mark["name"].removesuffix("_group")] = width["signal"]
        found |= width_signals(mark.get("marks", []))
    return found


spec = dashboard.to_dict()
vl_version = "_".join(alt.SCHEMA_VERSION.split(".")[:2])  # v6_4: the Vega-Lite this Altair writes specs for
vega = vl_convert.vegalite_to_vega(spec, vl_version=vl_version)
# dashboard.js widens each chart in GROW through its width signal (a KeyError here: a chart lost its name);
# charts sharing a signal must share alike.
widths = width_signals(vega["marks"])
grow = {}
for name, share in GROW.items():
    assert grow.setdefault(widths[name], share) == share, f"{name} shares its width with a chart that grows by another share"
# It also sets `dark`, ties the study start chart's height to the study lines', and fits the y axis of each chart in
# AXES, found by name.
assert set(AXES) <= set(widths), "a chart in AXES lost its name"
assert {"dark", "study_lines_height", "study_start_height", *AXES.values()} <= {
    signal["name"] for signal in vega["signals"]
}
# It lists the filters through their selections' stores, sets the Dates brush through timeline_x, and makes the
# filters chart as tall as its list.
assert set(FILTER_LABELS) == {selection.name for selection in FILTERS}
assert {f"{name}_store" for name in FILTER_LABELS} <= {data["name"] for data in vega["data"]}
assert "timeline_x" in {scale["name"] for scale in vega["scales"]}
assert {"filters_width", "filters_height"} <= {signal["name"] for signal in vega["signals"]}
# dashboard.js's guardBrush patches this handler of the Dates brush; without it, widening Dates would fail.
assert '{"signal": "dates_scale_trigger"}' in json.dumps(vega), "guardBrush in dashboard.js finds nothing"

page = Path(os.environ.get("DASHBOARD_HTML", "dashboard.html"))
partial = page.with_suffix(".tmp.html")  # still .html, so git ignores it too
# Inserted as values, so string.Template leaves the `$` in them alone.
page_css = Path("dashboard.css").read_text(encoding="utf-8")
page_js = Path("dashboard.js").read_text(encoding="utf-8")
assert "</style" not in page_css and "</script" not in page_js, "it would end its tag in the page"
partial.write_text(Template(Path("dashboard_template.html").read_text(encoding="utf-8")).substitute(
    page_css=page_css,
    page_js=page_js,
    vegaembed_script=vl_convert.javascript_bundle(vl_version=vl_version),
    spec=json.dumps(spec, cls=ScriptSafeJSON),  # a "</script>" in the data can't break the page
    embed_options=json.dumps({
        "mode": "vega-lite",
        # Vega runs the chart's expressions with its interpreter instead of eval, which the admin's
        # Content-Security-Policy for this page blocks.
        "ast": True,
        # The ⋯ menu offers only image downloads: "Open in Vega Editor" would send every row to vega.github.io.
        "actions": {"export": True, "source": False, "compiled": False, "editor": False},
    }),
    grow=json.dumps(grow),
    axes=json.dumps(AXES),
    filters=json.dumps(FILTER_LABELS),
    accent=BLUE,
    surface_light=SURFACE_LIGHT,
    surface_dark=SURFACE_DARK,
    ink_light=INK_LIGHT,
    ink_dark=INK_DARK,
), encoding="utf-8")
partial.replace(page)  # in one step, so nobody ever opens a half-written page